# IDSIA models

Three networks, one recipe. Train/val/test is Square–Random–Chirp runs 1–3 / run 4 / Melon. Windows are 50 steps at 100 Hz, stride 1, WeightedMSE $\lambda=0.1$, Adam $10^{-5}\to10^{-8}$ cosine $T=500$, 20 epochs.

Checkpoints: `out/runs/<family>/seed_k/{best.pt,last.pt}`. Training resumes `last.pt` if it exists.

In [ ]:
SEED = 0  # change to start or resume another copy
DEVICE = "cpu"  # train_device() will upgrade to mps when DEVICE is cpu and mps exists

## QuadDynamicsNet

**Input.** Ten stacked 13-D quotient frames $(g, v^{\mathrm{body}}, \omega, \Omega)$. No world $p$, no yaw.

**Output.** Body $(a,\alpha)\in\mathbb{R}^6$. Pose is kinematics: $p\mathrel{+}=v\Delta t+\tfrac12 Ra\Delta t^2$, $R\leftarrow R\exp((\omega+\tfrac12\alpha\Delta t)\Delta t)$.

**Symmetry (default on).** Four Crazyflie-X mirrors of the stack; accelerations are signed back and averaged (Reynolds).

In [ ]:
from quadrotor.fit import train_device
from quadrotor.idsia.recipe import QUAD_DYNAMICS
from quadrotor.idsia.train_lib import train_family

train_family(QUAD_DYNAMICS, SEED, device=train_device(DEVICE), symmetric=True)

## PhysQuotient

**Physics.** Frozen Crazyflie RK4 (published $\dot\omega=0$).

**Residual.** MLP reads the 13-D quotient at the current step and writes a 12-D $\Delta x$ in their normalized world coordinates (same update as published phys+res, without $p$/yaw as inputs).

In [ ]:
from quadrotor.idsia.recipe import PHYS_QUOTIENT

train_family(PHYS_QUOTIENT, SEED, device=train_device(DEVICE))

## PhysQuotientSym

Same frozen RK4. Residual reads the quotient and writes **body** $(\Delta p,\Delta v,\Delta\theta,\Delta\omega)$, orbit-averaged, then $R$ maps $\Delta p,\Delta v$ back to the world.

In [ ]:
from quadrotor.idsia.recipe import PHYS_QUOTIENT_SYM

train_family(PHYS_QUOTIENT_SYM, SEED, device=train_device(DEVICE))